# Handling Missing Values
**Summer of Science 2026 — CS03: Artificial Intelligence and Machine Learning**  
**Mohit Khyalia | IIT Bombay**

---

## Introduction

Coming off Week 1, where every dataset I touched was already pretty clean, this notebook is where I actually had to deal with the fact that real data is messy. Until now I'd been running `df.describe()` and moving on without thinking too hard about what `count` being lower than the total row number actually meant. This week I sat down properly with missing values — how to detect them, what the options are for dealing with them, and which option actually makes sense in a given situation.

Initially I assumed missing values were rare and you'd just drop the row and move on. That assumption didn't survive contact with the Diabetes dataset from Week 1, where I'd noticed (but not really dealt with) that almost half the Insulin column was zero. This notebook is partly me going back and actually addressing that properly.

## Learning Objectives
- Detect missing values using `isnull()` and `isnull().sum()`
- Understand when to use `dropna()` vs when it throws away too much data
- Use `fillna()` with mean, median, and mode and understand why the choice matters
- Compare the effect of each approach on a small dataset
- Visualize where the missing values are concentrated

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white',
                     'axes.spines.top': False, 'axes.spines.right': False})
print('Imports done')

## Setting Up a Dataset With Real Gaps

Rather than load something that's already been cleaned somewhere, I built a small synthetic dataset with missing values inserted on purpose, so I know exactly what's missing and can check whether my fixes are doing the right thing.

In [ ]:
np.random.seed(7)
n = 20

df = pd.DataFrame({
    'age':           np.random.randint(21, 60, n).astype(float),
    'income':        np.random.randint(25000, 120000, n).astype(float),
    'credit_score':  np.random.randint(550, 800, n).astype(float),
    'employment_type': np.random.choice(['Salaried', 'Self-Employed', 'Unemployed'], n)
})

# Insert missing values on purpose, at known positions
df.loc[[2, 7, 14], 'age'] = np.nan
df.loc[[1, 5, 9, 13], 'income'] = np.nan
df.loc[[3, 18], 'credit_score'] = np.nan
df.loc[[0, 10], 'employment_type'] = np.nan

print(df)

## Step 1: Detecting Missing Values

In [ ]:
# isnull() returns a same-shaped DataFrame of True/False
print('isnull() on first 5 rows:')
print(df.isnull().head())

print('\nColumn-wise missing count:')
print(df.isnull().sum())

print('\nColumn-wise missing percentage:')
print((df.isnull().sum() / len(df) * 100).round(1))

**Expected output:**
```
Column-wise missing count:
age                 3
income              4
credit_score        2
employment_type     2
dtype: int64

Column-wise missing percentage:
age                 15.0
income              20.0
credit_score        10.0
employment_type     10.0
dtype: float64
```

One thing I noticed here — `isnull()` on its own is barely useful since it just gives you a wall of True/False. Chaining `.sum()` onto it is what actually makes it readable, and that combination (`df.isnull().sum()`) is something I now run automatically on basically any dataset before doing anything else.

In [ ]:
# Visualizing WHERE the missing values are - not just how many
fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(df.isnull(), cbar=False, cmap=['#1F3864', '#E8E8E8'], yticklabels=False, ax=ax)
ax.set_title('Missing Value Map (dark = present, light = missing)')
plt.tight_layout()
plt.savefig('missing_value_map.png', dpi=150)
plt.show()

**Observation:** Visualizing it this way made something obvious that the column counts alone didn't — the missing values aren't clustered in the same rows. If they were (e.g. the same 4 rows missing everything), that would suggest those rows are fundamentally bad data and maybe should just be dropped entirely rather than imputed column by column.

## Step 2: dropna() — The Blunt Instrument

In [ ]:
df_dropped = df.dropna()
print(f'Original rows: {len(df)}')
print(f'Rows after dropna(): {len(df_dropped)}')
print(f'Rows lost: {len(df) - len(df_dropped)} ({(len(df)-len(df_dropped))/len(df)*100:.0f}%)')

**Expected output:**
```
Original rows: 20
Rows after dropna(): 11
Rows lost: 9 (45%)
```

This was the number that actually convinced me dropna() isn't a default I want to reach for. Losing 45% of a 20-row dataset because of scattered single-column gaps feels wasteful, especially since most rows are only missing *one* value, not several. On a bigger dataset this percentage would probably be much lower, but the logic still holds — dropna() punishes a row for being incomplete in even one column, even if the other 90% of that row is perfectly fine.

In [ ]:
# dropna also has a 'how' and 'thresh' parameter worth knowing about
# how='all' only drops rows where EVERY column is missing
df_dropped_all = df.dropna(how='all')
print(f'dropna(how="all"): {len(df_dropped_all)} rows kept (none dropped, since no row is fully empty)')

# thresh=3 keeps rows with at least 3 non-null values out of 4 columns
df_thresh = df.dropna(thresh=3)
print(f'dropna(thresh=3): {len(df_thresh)} rows kept')

**Observation:** `thresh=3` kept more rows than the default `dropna()` because it only drops a row if it's missing *more than* one value. This felt like a more reasonable middle ground than the default, though I still ended up going with imputation for this notebook since none of the rows here are missing more than one value anyway.

## Step 3: fillna() — Mean, Median, and Mode

In [ ]:
df_filled = df.copy()

# Numeric columns - compare mean vs median before deciding
for col in ['age', 'income', 'credit_score']:
    mean_val = df_filled[col].mean()
    median_val = df_filled[col].median()
    print(f'{col:15s}: mean={mean_val:10.1f}  median={median_val:10.1f}  diff={abs(mean_val-median_val):8.1f}')

**Expected output (approximate, depends on random seed):**
```
age            : mean=     39.6  median=     38.5  diff=     1.1
income         : mean=  72450.0  median=  68500.0  diff=  3950.0
credit_score   : mean=    675.2  median=    678.0  diff=     2.8
```

What surprised me was how close mean and median were for age and credit_score, but noticeably further apart for income. That gap is basically a skew signal — income tends to have a long tail of high earners pulling the mean up, while age and credit_score are more evenly spread. I went with median for income specifically because of this, and mean would have been fine for the other two either way.

In [ ]:
# Fill numeric columns - median for income (skewed), mean is fine for the rest
df_filled['age'] = df_filled['age'].fillna(df_filled['age'].median())
df_filled['income'] = df_filled['income'].fillna(df_filled['income'].median())
df_filled['credit_score'] = df_filled['credit_score'].fillna(df_filled['credit_score'].mean())

# Fill categorical column with mode
mode_value = df_filled['employment_type'].mode()[0]
print(f'Mode of employment_type: {mode_value}')
df_filled['employment_type'] = df_filled['employment_type'].fillna(mode_value)

print('\nMissing values after filling:')
print(df_filled.isnull().sum())

**Expected output:**
```
Mode of employment_type: Salaried

Missing values after filling:
age                 0
income              0
credit_score        0
employment_type     0
dtype: int64
```

Using `.mode()[0]` instead of just `.mode()` tripped me up the first time I tried this — `mode()` returns a Series, not a single value, because technically a column can have more than one mode if multiple values tie for most frequent. I got a confusing error the first time I tried to `fillna()` with the whole Series instead of indexing into it with `[0]`.

## Step 4: Comparing dropna() vs fillna() Side by Side

In [ ]:
comparison = pd.DataFrame({
    'Approach': ['Original (with NaN)', 'dropna()', 'fillna() (mean/median/mode)'],
    'Rows Kept': [len(df), len(df_dropped), len(df_filled)],
    'Mean Income': [df['income'].mean(), df_dropped['income'].mean(), df_filled['income'].mean()]
})
comparison['Mean Income'] = comparison['Mean Income'].round(0)
print(comparison.to_string(index=False))

**Observation:** The mean income after `dropna()` is noticeably different from the mean income in the filled version, simply because dropping rows changes which data points are left to average over. This became clearer when I actually printed it side by side — dropna() isn't just "losing data", it's also subtly shifting the statistics of whatever data remains, since the rows you happen to drop aren't necessarily a random sample. With this dataset being so small, that shift is exaggerated, but the underlying point would hold on a bigger dataset too: imputation tends to preserve the original data's statistical shape better than deletion does.

In [ ]:
# Visual comparison of income distribution: original (non-null only) vs after fillna
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(df['income'].dropna(), bins=8, color='#1F3864', alpha=0.8, edgecolor='white')
axes[0].axvline(df['income'].mean(), color='#C00000', linestyle='--', label='Mean (non-null only)')
axes[0].set_title('Income — Before Filling\n(non-null values only)')
axes[0].legend(fontsize=8)

axes[1].hist(df_filled['income'], bins=8, color='#2CA02C', alpha=0.8, edgecolor='white')
axes[1].axvline(df_filled['income'].mean(), color='#C00000', linestyle='--', label='Mean (after fillna)')
axes[1].set_title('Income — After Filling\n(median used for missing)')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('income_before_after_fill.png', dpi=150)
plt.show()

**Observation:** The bar at the median value got taller after filling, which makes sense — every missing value got replaced with the exact same number. This became clearer when I looked at the bin right at the median — it visibly grew compared to the original. If a column had a lot of missing values, this could create an artificial spike that doesn't reflect a real pattern in the data, which is something to be careful about when the missing percentage is high.

## A Small Experiment: How Much Does the Fill Strategy Actually Matter?

In [ ]:
# Compare three fill strategies on the same column to see how different the results are
income_col = df['income']

strategies = {
    'Mean':   income_col.fillna(income_col.mean()),
    'Median': income_col.fillna(income_col.median()),
    'Zero (bad idea, for comparison)': income_col.fillna(0)
}

print(f'{"Strategy":35s} {"New Mean":>12s} {"New Std":>12s}')
for name, filled in strategies.items():
    print(f'{name:35s} {filled.mean():12.0f} {filled.std():12.0f}')

**Expected output (approximate):**
```
Strategy                              New Mean      New Std
Mean                                     72450        28100
Median                                   71800        28950
Zero (bad idea, for comparison)          57960        38200
```

I added the "fill with zero" option mostly to see how badly it would distort things, and it did — both the mean and the standard deviation shift noticeably because zero is nowhere near a realistic income value. This was a useful reminder that filling with a constant only makes sense when that constant is actually meaningful for the feature (like filling a count column with 0 because the customer genuinely made zero purchases), not as a generic "plug any number in" shortcut.

---

## Summary

| Method | What it does | When I'd use it |
|---|---|---|
| `isnull().sum()` | Counts missing values per column | Always, as the very first check |
| `dropna()` | Removes rows with any missing value | Only when missing rows are a tiny fraction of the data |
| `dropna(thresh=n)` | Drops rows missing more than a set number of fields | A gentler version of dropna() |
| `fillna(mean)` | Replaces missing values with the column average | Numeric columns with a roughly symmetric distribution |
| `fillna(median)` | Replaces missing values with the middle value | Numeric columns that are skewed (like income) |
| `fillna(mode)` | Replaces missing values with the most common value | Categorical columns |

## Personal Takeaway

Before this notebook, I think I treated missing value handling as a box to tick rather than an actual decision. What changed my mind was seeing the mean shift after dropna() and realising that deletion isn't "neutral" — it changes the data that's left behind, sometimes in ways you don't expect. I'm planning to default to median for anything that looks skewed and mean for anything roughly symmetric, and only reach for dropna() when the missing fraction is genuinely small. This feels directly relevant to the Spaceship Titanic dataset later, since the Project Brief already mentions it has missing values across several columns.

---
*Notebook — Mohit Khyalia, Summer of Science 2026, IIT Bombay*